# 02. Features and the ragged edge

A nowcast for quarter Q at horizon h is made at the end of month T = first month of Q + h:

| h | origin | typical information |
|---|---|---|
| -1 | end of the month before the quarter | nothing about Q itself |
| 0 | end of month 1 | financial data for month 1; output data for the previous quarter |
| 1 | end of month 2 | retail and prices for month 1 |
| 2 | end of month 3 | output data for month 1 of Q |
| 3 | end of the month after Q | output data for months 1 and 2 of Q |

Each indicator is published with a different lag, so the most recent month available differs across
series: the **ragged edge**. Features are the six latest available monthly values of each indicator
(U-MIDAS: unrestricted mixed-data sampling), so one model per horizon has consistently aligned
features.

In [1]:
import sys, warnings, pickle
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (9, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
T = ROOT / "outputs" / "tables"; FIG = ROOT / "outputs" / "figures"
from nowcast.config import HORIZON_LABELS

In [2]:
from nowcast.features import build_all, main_spec, feature_names, origin_month
from nowcast.config import INDICATORS
B = build_all([-1, 0, 1, 2, 3])
names = feature_names()
main = [n for n in names if main_spec(n)]
print(f"features in the main specification: {len(main)} "
      f"({len(main) - 2} indicator lags + 2 GDP lags); training samples 31-107 quarters")

features in the main specification: 236 (234 indicator lags + 2 GDP lags); training samples 31-107 quarters


## Which months are known at each horizon? (Q3 2025 as an example)

In [3]:
q = pd.Period("2025Q3", "Q")
rows = []
for ind in [i for i in INDICATORS if main_spec(f"{i.code}_l0")]:
    rows.append({"indicator": ind.name, "group": ind.group,
                 **{HORIZON_LABELS[h]: str(origin_month(q, h) - ind.lag) for h in [-1, 0, 1, 2, 3]}})
tab = pd.DataFrame(rows).drop_duplicates("group").set_index("indicator")
print("Latest month available for one indicator per group:")
tab

Latest month available for one indicator per group:


,group,Before quarter,End of month 1,End of month 2,End of month 3,Month after quarter
indicator,,,,,,
Index of Services (real time),Output (real time),2025-04,2025-05,2025-06,2025-07,2025-08
Retail sales volume,Retail,2025-05,2025-06,2025-07,2025-08,2025-09
Goods exports volume,Trade,2025-04,2025-05,2025-06,2025-07,2025-08
Unemployment rate,Labour market,2025-03,2025-04,2025-05,2025-06,2025-07
CPI index,Prices,2025-05,2025-06,2025-07,2025-08,2025-09
Bank Rate,Money and markets,2025-06,2025-07,2025-08,2025-09,2025-10


## The live information set: data published by the end of October 2026

In [4]:
live = {"2026Q3": 3, "2026Q4": 0}
for qs, h in live.items():
    x = B["X"][h].loc[qs, [c for c in main if c.endswith("_l0") or c.startswith("GDP_")]]
    print(f"{qs} at horizon {h} ({HORIZON_LABELS[h]}): missing latest values ->",
          [c for c in x.index if np.isnan(x[c])] or "none")
B["X"][3].loc["2026Q3", ["RT_IOS_l0", "RT_IOS_l1", "RT_IOP_l0", "RT_IOP_l1", "J5EK_l0", "GDP_q1"]].round(2)

2026Q3 at horizon 3 (Month after quarter): missing latest values -> none
2026Q4 at horizon 0 (End of month 1): missing latest values -> ['GDP_q1']


RT_IOS_l0    0.00
RT_IOS_l1    0.50
RT_IOP_l0    0.80
RT_IOP_l1    0.30
J5EK_l0     -0.39
GDP_q1       0.53
Name: 2026Q3, dtype: float64

`RT_IOS_l0` at horizon 3 is August 2026 services growth as published in October 2026; `GDP_q1` is
Q2 2026 growth as known in the latest GDP vintage before the nowcast.

## Correlation structure
Is this a highly collinear panel? Check the pairwise correlations and the conditioning of the
standardised design (1998Q1-2019Q4, end-of-quarter horizon).

In [5]:
from nowcast.models import Standardiser
X = B["X"][2][main].loc["1998Q1":"2019Q4"]
Z = Standardiser().fit(X.values).transform(X.values)
C = np.corrcoef(Z.T)
off = np.abs(C[np.triu_indices_from(C, 1)])
off = off[np.isfinite(off)]
print(f"pairs of features with |corr| > 0.5: {(off > 0.5).mean():.1%}; > 0.8: {(off > 0.8).mean():.1%}")
print(f"condition number of the standardised design: {np.linalg.cond(Z):.2e}")
print(f"rows (quarters) x columns (features): {Z.shape}")

pairs of features with |corr| > 0.5: 0.8%; > 0.8: 0.2%
condition number of the standardised design: 8.50e+15
rows (quarters) x columns (features): (88, 236)


Pairwise correlation is modest: monthly growth rates are noisy, and successive lags of the same
indicator are only weakly correlated. The design is nonetheless singular, because there are
more features than quarters. OLS then has infinitely many exact-fit solutions. That, not
collinearity, is the main reason this problem needs regularisation (learn/01 and learn/02).